# Spin BuAli — radiology pipeline benchmark

Hands English-dominant radiology dictations (Persian words mixed in) straight
to an **audio-capable language model** — no transcription step — and scores
the report it produces against the radiologist's signed one.

**Every run is its own cell**, and each one writes its own CSV before the cell
finishes. Stop the session whenever you like — nothing already run needs to be
redone, because nothing here is a loop.

**All the logic lives in the repo, not in this notebook.** Cell 2 clones it;
every cell after that imports from it. Fixing a bug means fixing the `.py`
file, pushing it, and re-running cell 2 — never re-uploading this file.

**Before running:** Settings → Accelerator **GPU T4 ×2**, Internet **On**, and
attach the `spin-buali-dataset` dataset under *Add Input*.

## 1 — Check the hardware

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv

## 2 — Get the code

Clones `https://github.com/Spinc-AI/Spin_BuAli.git` (branch `benchmark-selfcontained`) into `/kaggle/working/Spin_BuAli`. If
it is already there — because this is not the first cell run this session —
it pulls instead, so re-running this cell after a code fix upstream is enough
to pick it up.

In [ ]:
import pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/kaggle/working/Spin_BuAli")

if REPO_DIR.exists():
    print("repo already present -- pulling the latest commit")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "benchmark-selfcontained"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/benchmark-selfcontained"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "benchmark-selfcontained", "--depth", "1",
                    "https://github.com/Spinc-AI/Spin_BuAli.git", str(REPO_DIR)], check=True)


# Only benchmark/ goes on the path directly. Its own bridge.py reaches
# evaluation/, controller/ and stt/app/ itself, appending each to the END of
# sys.path rather than the front -- three of those folders each have their own
# config.py, and inserting all of them up front here would make the wrong one
# win depending on loop order, exactly the bug this split avoids.
sys.path.insert(0, str(REPO_DIR / "benchmark"))

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print(f"\nrunning commit {commit}")

In [ ]:
# transformers >= 5.5.0 is REQUIRED, not opportunistic. gemma-4-* is
# model_type "gemma4", which only exists from 5.5.0 (added 2026-04-02);
# on Kaggle's preinstalled 4.x, AutoConfig.from_pretrained() raises a bare
# KeyError('gemma4') before reaching any model code. core_llm/ is already
# written against v5's API anyway -- `dtype=` (v4 called it `torch_dtype`)
# and AutoModelForMultimodalLM -- so this aligns the runtime with the code.
# v5 also drops TensorFlow/JAX entirely, removing a whole class of
# numpy-related import crashes.
#
# --no-deps on the HF stack is the important part. A plain
# `pip install --upgrade transformers` pulls numpy up with it, and numpy
# >= 2.1 changed numpy._core.umath's internals, which breaks the torch
# wheel this image was built against -- first as "cannot import name 'nn'
# from partially initialized module 'torch'", then, after trying to repair
# it, as "cannot import name '_center' from numpy._core.umath". Both were
# hit live. Installing the HF packages without their dependency closure
# leaves numpy and torch exactly as the image shipped them, which is the
# only state they are known to work in. tokenizers/huggingface-hub/
# safetensors come along because v5 needs newer ones than 4.x shipped.
!pip install -q --no-deps --upgrade "transformers>=5.5.0" tokenizers huggingface-hub safetensors
!pip install -q python-dotenv sentencepiece bitsandbytes accelerate
# Two audio front-ends that are not optional extras:
#   mistral-common[audio]  Voxtral tokenizes through mistral-common, not a
#                          Jinja template, and its processor is what opens the
#                          file named by the chat message's `path`.
#   librosa                Qwen2-Audio's processor does NOT load audio itself;
#                          the waveform is decoded and resampled to 16 kHz by
#                          the caller (core_llm/model.py's Qwen2AudioModel).
# Installed WITH their dependencies, unlike the HF stack above -- neither
# pins numpy, and the version cell below is what catches it if that changes.
!pip install -q "mistral-common[audio]" librosa

In [ ]:
# Fail here, loudly, rather than twenty minutes into a run. Every version
# problem this notebook has hit would have been one line of output instead
# of a stack trace in a model loader.
import numpy, torch, transformers

print(f"transformers {transformers.__version__}")
print(f"torch        {torch.__version__}")
print(f"numpy        {numpy.__version__}")

_major = int(transformers.__version__.split(".")[0])
_minor = int(transformers.__version__.split(".")[1])
if (_major, _minor) < (5, 5):
    raise SystemExit(
        f"transformers {transformers.__version__} cannot load gemma-4-* "
        "(model_type 'gemma4' needs >= 5.5.0). The install cell above did "
        "not take -- restart the kernel and run it again before continuing.")
print("\nversions OK")

## 3 — Imports and hardware placement

`tiers.py` decides, for the language models, the highest precision this
hardware can hold: fp16 if it fits, otherwise int8, otherwise 4-bit. A model
placed at anything other than fp16 is tier B — its score includes whatever the
compression cost.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import bridge, dataset, kaggle_dataset, leaderboard, llm, pipeline
import plan, report_structure, runner, scoring, tiers, transcribe

import pandas as pd

usable_gb, cards = tiers.usable_vram()
print(f"{cards} GPU(s), ~{usable_gb:.1f} GB usable each "
      f"(~{usable_gb * max(cards, 1):.1f} GB if a model is sharded)\n")

placements = tiers.plan_placements(plan.LLM_PARAMS, usable_gb, cards)
print(f"{'model':18} {'tier':5} {'placement'}")
for p in placements:
    print(f"  {p.model:18} {p.tier:5} {p.describe()}")

## 4 — Hugging Face authentication

Several checkpoints are gated: HF serves them only to an account that has
accepted the model's licence on its page. Without a token, or without having
accepted the licence, the load fails with *"Cannot access gated repo"* — which
is easy to mistake for a bug here.

**Typed live, not written into this notebook.** This cell prompts for the
token with a masked field — nothing is echoed back, and nothing here prints
it — so it never ends up in this cell's source or its saved output, even if
the notebook is public. Re-run this cell each session; there is nothing to
carry over because nothing was saved.

If this notebook is your own and stays private, **Add-ons → Secrets** (a
secret named `HF_TOKEN`) is one step less per session — this cell tries that
first and only prompts if no secret is set. A public notebook should rely on
the prompt, not a secret attached to the notebook.

Get a token (read scope is enough) at huggingface.co/settings/tokens.

In [ ]:
import getpass
import os
from huggingface_hub import login, model_info, whoami


def resolve_hf_token():
    """A Kaggle secret if one is set, otherwise a live masked prompt.

    Neither path writes the token anywhere this notebook file can carry it:
    a secret lives in Kaggle's own store, not in the notebook, and getpass
    does not echo what is typed and nothing here prints it back.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("HF_TOKEN")
        if secret:
            return secret.strip(), "Kaggle secret HF_TOKEN"
    except Exception:
        pass
    for name in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
        if os.environ.get(name):
            return os.environ[name].strip(), f"${name}"
    typed = getpass.getpass("Hugging Face token (hidden while typing, not saved anywhere): ")
    return (typed.strip(), "typed just now") if typed.strip() else (None, None)


_token, _source = resolve_hf_token()
if _token:
    login(token=_token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_HUB_TOKEN"] = _token
    name = whoami().get("name", "unknown")
    print(f"signed in as {name}   (token from {_source})")
else:
    print("No token given. Ungated models still download; gated ones will not.")

In [ ]:
# Checked once, here, instead of discovering each blocked checkpoint hours
# apart inside a run cell.
def check_access(repo_ids):
    reachable, blocked = [], []
    for repo in sorted(set(repo_ids)):
        try:
            model_info(repo, token=_token or None)
            reachable.append(repo)
        except Exception as error:
            text = str(error).lower()
            if "gated" in text or "awaiting" in text:
                reason = "gated -- accept the licence on its model page"
            elif "401" in text or "not found" in text:
                reason = "not found, or private to another account"
            else:
                reason = type(error).__name__
            blocked.append((repo, reason))
    return reachable, blocked


# Only what this notebook actually loads. Checking every registered STT
# engine too would report a blocked checkpoint for a model no cell here runs,
# which reads as a problem and is not one.
llm_repos = [llm._hugging_face_id(key) for key in runner.MULTIMODAL_LLM]

reachable, blocked = check_access(llm_repos)
print(f"{len(reachable)} of {len(llm_repos)} checkpoints reachable")
for repo, reason in blocked:
    print(f"  BLOCKED  {repo}")
    print(f"           {reason}   ->   https://huggingface.co/{repo}")
if blocked:
    print("\nAccept the licence on each page above, signed in as the same account,")
    print("then re-run this cell.")

## 5 — Dataset

Searched at any depth under `/kaggle/input`, because how deep `labels.csv`
sits depends on how the dataset was zipped. If this cell says the file was not
found, it prints the mounted tree — paste the path it shows into
`LABELS_OVERRIDE` below and re-run.

In [ ]:
LABELS_OVERRIDE = None   # e.g. "/kaggle/input/spin-buali-dataset/Small_Demo/labels.csv"

try:
    LABELS = kaggle_dataset.resolve(override=LABELS_OVERRIDE)
except FileNotFoundError as error:
    raise SystemExit(str(error))

DATA_DIR = LABELS.parent
labels = pd.read_csv(LABELS)

print(f"labels : {LABELS}")
print(f"folder : {DATA_DIR}")
print(f"audio  : {kaggle_dataset.count_audio(DATA_DIR)} file(s)")
print(f"rows   : {len(labels)}")
labels[["asset_id", "audio", "report"]].head()

In [ ]:
clips = dataset.from_csv(LABELS)
census = dataset.describe(clips)
assert not census["missing_audio"], census["missing_audio"]
print(f"{census['items']} recording(s), {census['labelled']} labelled")

for item in clips:
    audio, sr = dataset.load_audio(item.audio)
    print(f"  {item.asset_id:12} {len(audio) / sr:6.1f}s")

## 6 — Run configuration

Shared settings every run cell below reads -- the report-structure addendum,
the token cap, and where results land. The roster itself is fixed
(`runner.MULTIMODAL_LLM`); see cell 7 for what is in it and why.

**The addendum is not part of `controller/prompts.py`.** `report_structure.GUIDE`
(`SECTION_ORDER`, `BOILERPLATE_ANCHORS`) was *read off* the nine reference
reports, but what it encodes -- organ order, house phrasing -- is a real,
standing expectation the radiologists have for every report, not an artefact
of this one dataset. Telling the model that expectation up front is what
production should do too; it is on by default for that reason.

Every result row still stamps `structure_guided` (`True`/`False`), so a run
made without it (`STRUCTURE_GUIDE = None`, to see how the model does with no
house-style hint at all) is always distinguishable in a master CSV from one
made with it -- useful for comparison, not because either condition is
somehow illegitimate.

In [ ]:
# LANGUAGE, DEVICES and PREPROCESSING only reach the STT stage, which a
# multimodal run does not have -- they are kept so a `separate` cell can be
# pasted back in unchanged, and because PREPROCESSING still names the run.
LANGUAGE = "en"                # the dictation is English-dominant, Persian words mixed in
DEVICES = None                 # None = whichever card has the most room when the run starts
PREPROCESSING = "adaptive"     # None | "fixed" | "uniform" | "adaptive" | "adaptive-vad"

STRUCTURE_GUIDE = report_structure.GUIDE   # None scores the bare controller prompt -- see cell 6's note above

# The prompt asks for three full fields (raw_transcript, corrected_transcript,
# final_text) -- easy to overrun the 1536-token default on a real report. A
# generation cut off mid-JSON shows up as "no JSON object found" or a
# JSONDecodeError, and only on the longer clips, since the model never reached
# the closing brace. Raise this if a run shows that pattern.
#
# It is NOT the fix for the other failure in the last round -- a model that
# repeats one sentence until it runs out of budget ("There is a large left
# side of the stone." x60). More tokens just buys more repetition; that one is
# the model, not the cap.
MAX_NEW_TOKENS = None          # None uses settings.LLM_MAX_NEW_TOKENS (1536); try 3072 if truncating

RESULTS_DIR = pathlib.Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def placement_for(llm_key):
    """This LLM's precision/card count from cell 3's placement table --
    looked up per run cell instead of once, since every cell here can name a
    different LLM."""
    p = next(p for p in placements if p.model == llm_key)
    return p.precision, p.cards

## 7 — Runs: 4 audio-capable LLMs, `multimodal` only

**`separate` is not in this notebook.** It is still a real pipeline in
`benchmark/pipeline.py` and `controller/pipelines.py` — this notebook just
doesn't run it. On this dataset the STT stage was the whole result: seamless
returned text like *"the graphics card has a one-to-two-mile radius in
Madrid"* for a kidney ultrasound, and every LLM downstream then scored near
WER 1.0 for correctly refusing to invent a report out of noise. That measures
one STT engine's failure, not the language models. Re-add those cells when
there is an STT engine worth putting in front of them — nothing in the repo
was removed.

So every cell here is one **`multimodal`** run: the LLM is handed the
recording itself and produces the report in a single call, with no
transcription step anywhere. `runner.MULTIMODAL_LLM` is the roster —
every audio-capable model this hardware can hold at some precision, lightest
first:

* `voxtral-mini-3b`
* `gemma-4-e4b`
* `qwen2-audio-7b`
* `gemma-4-12b`

Three vendors on purpose (Mistral, Google, Alibaba). One vendor's audio
front-end across every row would make a family-wide weakness look like a
property of the task.

**Precision comes from cell 3's tier table, and is now actually applied.**
Previously a model placed at `int8` still loaded at full precision — the
adapter stored the placement and never used it — so gemma-4-12b put ~24 GB of
fp16 weights onto a 14.56 GB card, failed, and stamped the CSV `int8` anyway.
A placement that cannot be honoured now raises instead.

**`phi-4-multimodal` is deliberately absent.** It cannot currently load on
this environment — confirmed across five rounds of real fixes (missing pip
deps, a stale import, a `from_pretrained` kwarg its own config class ignores,
a `flash_attn` dependency worked around via eager attention), ending on a
meta-tensor incompatibility inside its own vendor code that no caller-side fix
resolves. Its checkpoint stays registered in `core_llm/model.py` for whoever
eventually resolves this; see `runner.py`'s comment above `TOP3_LLM`.

**4 runs, 4 cells.**

**To add a run:** copy a cell and change its `llm_key` and `label`. Every cell
is independent — stopping the session after any of them loses nothing.

**There is no transcript cache here**, because there is no STT stage to cache.
Each cell does its full work from the audio. (The cache in
`RESULTS_DIR/transcripts/` only ever serves `separate`.)

**Memory is cleared at the start of every run, not just the end.** Each cell
prints free VRAM per card before it allocates. A run that dies mid-load
leaves its partial weights pinned by the traceback -- which `unload()` can
never reach, because the model was never assigned anywhere -- so the cleanup
drops those references (`sys.last_traceback`, and the notebook's `Out`/`_`
history) before collecting. To do the same by hand after a failure, without
starting another run: `runner.free_vram()`. If a card still shows as more
than half used after that, something outside this process holds it and only
a kernel restart will clear it -- the cell says so when it happens.

### 7.1 — `voxtral-mini-3b`

mistralai/Voxtral-Mini-3B-2507 — 4.7B. A Whisper-large-v3 encoder in front of Ministral 3B. The only model here that fits one card at fp16.

In [ ]:
PRECISION, CARDS = placement_for("voxtral-mini-3b")
df_01 = runner.run_one(
    None, "voxtral-mini-3b", "multimodal", clips,
    language=LANGUAGE, devices=DEVICES, precision=PRECISION, cards=CARDS,
    preprocessing=PREPROCESSING, structure_guide=STRUCTURE_GUIDE, results_dir=RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
    label="01_multimodal__voxtral-mini-3b__" + str(PREPROCESSING),
)
df_01[df_01["asset_id"] != "SUMMARY"][
    ["asset_id", "wer", "medical_term_f1", "negation_errors",
     "laterality_errors", "number_errors", "requires_medical_review"]]

### 7.2 — `gemma-4-e4b`

google/gemma-4-E4B-it — 7.85B, encoder-free ("Unified") audio. The "E4B" is effective compute, not the on-disk parameter count.

In [ ]:
PRECISION, CARDS = placement_for("gemma-4-e4b")
df_02 = runner.run_one(
    None, "gemma-4-e4b", "multimodal", clips,
    language=LANGUAGE, devices=DEVICES, precision=PRECISION, cards=CARDS,
    preprocessing=PREPROCESSING, structure_guide=STRUCTURE_GUIDE, results_dir=RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
    label="02_multimodal__gemma-4-e4b__" + str(PREPROCESSING),
)
df_02[df_02["asset_id"] != "SUMMARY"][
    ["asset_id", "wer", "medical_term_f1", "negation_errors",
     "laterality_errors", "number_errors", "requires_medical_review"]]

### 7.3 — `qwen2-audio-7b`

Qwen/Qwen2-Audio-7B-Instruct — 8.4B. Trained explicitly for instruction-following over audio, not just transcription.

In [ ]:
PRECISION, CARDS = placement_for("qwen2-audio-7b")
df_03 = runner.run_one(
    None, "qwen2-audio-7b", "multimodal", clips,
    language=LANGUAGE, devices=DEVICES, precision=PRECISION, cards=CARDS,
    preprocessing=PREPROCESSING, structure_guide=STRUCTURE_GUIDE, results_dir=RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
    label="03_multimodal__qwen2-audio-7b__" + str(PREPROCESSING),
)
df_03[df_03["asset_id"] != "SUMMARY"][
    ["asset_id", "wer", "medical_term_f1", "negation_errors",
     "laterality_errors", "number_errors", "requires_medical_review"]]

### 7.4 — `gemma-4-12b`

google/gemma-4-12B-it — 12B, the largest audio-capable Gemma 4. Runs quantized here; its score includes whatever that costs.

In [ ]:
PRECISION, CARDS = placement_for("gemma-4-12b")
df_04 = runner.run_one(
    None, "gemma-4-12b", "multimodal", clips,
    language=LANGUAGE, devices=DEVICES, precision=PRECISION, cards=CARDS,
    preprocessing=PREPROCESSING, structure_guide=STRUCTURE_GUIDE, results_dir=RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
    label="04_multimodal__gemma-4-12b__" + str(PREPROCESSING),
)
df_04[df_04["asset_id"] != "SUMMARY"][
    ["asset_id", "wer", "medical_term_f1", "negation_errors",
     "laterality_errors", "number_errors", "requires_medical_review"]]

## 8 — Master results

Merges the SUMMARY row of every `results__*.csv` on disk — whichever cells
above have actually been run — sorted by corpus WER. Safe to run after any
subset of the cells above, and safe to re-run after more of them finish.

In [ ]:
master = runner.build_master(RESULTS_DIR)
master

**Read it in this order.** Corpus WER is templated-text noise as much as
signal on this dataset — a wrong patient's report can score a better WER than
a correctly reworded one. Negation, laterality, number and unit error rates
are what actually separate a usable configuration from one that changes what
the report means.

## 9 — Save

Every run already wrote its own CSV as its cell finished. This only lists
them. Download the folder before ending the session — Kaggle does not keep
`/kaggle/working` between sessions unless this notebook version is saved with
its output.

To resume later without redoing a finished run: re-attach the downloaded CSVs
into `/kaggle/working/results` (or a fresh Kaggle Dataset) before running
`build_master`, and only re-run the run cells you have not done yet.

In [ ]:
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {path.name:52} {path.stat().st_size / 1024:8.1f} KB")